# RAG: Multi-Source Retrieval

This notebook demonstrates querying multiple independent vector database collections
through a unified retrieval interface with source provenance preserved.
Run example_RAG_01_load.ipynb first to have the base data available.

## Initialize

In [ ]:
from agentic_patterns.core.agents import get_agent, run_agent
from agentic_patterns.core.config.config import MAIN_PROJECT_DIR
from agentic_patterns.core.doc_ingestion.models import DocumentProvenance
from agentic_patterns.core.vectordb import get_vector_db
from agentic_patterns.core.vectordb.multi_source import MultiSourceRetriever

In [ ]:
DOCS_DIR = MAIN_PROJECT_DIR / "tests" / "data" / "books"
SCIFI_TITLES = {"hhgttg", "foundation"}

## Ingest into two separate collections

We split the book corpus into two named collections based on genre.
Each collection uses the same embedding model but contains different documents.

In [ ]:
vdb_scifi = get_vector_db("books_scifi")
vdb_other = get_vector_db("books_other")

for txt_file in sorted(DOCS_DIR.glob("*.txt")):
    vdb = vdb_scifi if txt_file.stem in SCIFI_TITLES else vdb_other
    provenance = DocumentProvenance(original_file=txt_file, source=txt_file.stem)
    added = vdb.ingest_file(txt_file, provenance, force=False)
    coll_name = "scifi" if txt_file.stem in SCIFI_TITLES else "other"
    print(f"{txt_file.name} -> {coll_name} ({added} chunks added)")

print(f"\nscifi collection: {vdb_scifi.count()} chunks")
print(f"other collection: {vdb_other.count()} chunks")

## Multi-source retrieval

MultiSourceRetriever queries all sources in parallel and merges results.
Each document's metadata includes 'source_collection' identifying its origin.

In [ ]:
retriever = MultiSourceRetriever(
    sources={
        "scifi": vdb_scifi,
        "other": vdb_other,
    }
)

query = "Who is Zaphod?"
results = retriever.retrieve_all(query=query, max_results=5)

for doc in results:
    source = doc.metadata.get("source_collection", "unknown")
    print(f"[{source}] score={doc.score:.3f} | {doc.text[:100]}...")

## RAG with source attribution

Each passage is prefixed with its source collection so the LLM can cite it.

In [ ]:
context_blocks = []
for doc in results:
    source = doc.metadata.get("source_collection", "unknown")
    context_blocks.append(f"[{source}]\n{doc.text}")

context = "\n\n---\n\n".join(context_blocks)

prompt = f"""Answer the question using the sources below.
For each claim, cite the source collection in brackets (e.g. [scifi], [other]).

## Sources

{context}

## Question

{query}
"""

agent = get_agent()
agent_run, _ = await run_agent(agent, prompt=prompt, verbose=True)
print(f"\nAnswer:\n{agent_run.result.output}")

## Cross-domain query

A query that spans both collections shows how provenance helps distinguish sources.

In [ ]:
cross_query = "How do characters cope with unexpected journeys?"
cross_results = retriever.retrieve_all(query=cross_query, max_results=6)

sources_seen = set()
for doc in cross_results:
    source = doc.metadata.get("source_collection", "unknown")
    sources_seen.add(source)
    print(f"[{source}] {doc.text[:80]}...")

print(f"\nSources retrieved from: {sources_seen}")